# TabPFN vs. Week 5 on Churn

**Week 6 · Lesson 9** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week6/lectures/09_tabpfn_vs_week5_churn.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Adapted from Hollmann et al., "Accurate predictions on small data with a tabular
foundation model," Nature 637 (2025), and the [TabPFN](https://github.com/PriorLabs/TabPFN)
documentation.*

**Built with PriorLabs-TabPFN.** This notebook uses the open TabPFN v2 classifier
weights, released under the
[Prior Labs License](https://huggingface.co/Prior-Labs/TabPFN-v2-clf/blob/main/LICENSE.txt)
(Apache 2.0 with an attribution requirement).

## Setup

Same boilerplate as every notebook this term — see Week 1 Lesson 8 if this is your
first time seeing it.

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

## 1. Business Problem

**Business question:** Lesson 8 ended on a claim — TabPFN, a network pretrained once
on synthetic tables, beats tuned trees on business-sized tables with no training on
your data at all. On our churn data, does it rank customers by churn risk better than
Week 5's tuned XGBoost — and if it does, should the retention team switch?

**Why it matters:** the retention team calls customers in order of predicted risk, so
a better *ranking* means fewer wasted calls. But the model that ranks best isn't
automatically the one to run: Week 5's XGBoost is already built, runs in
milliseconds, and has SHAP explanations (Week 5 Lessons 10–11). A challenger has to
earn its license, compute, and explanation costs on this exact question, held to the
same test set as every prior model.

Lesson 8's rule applies in both directions: don't dismiss TabPFN because simple
models usually win on tables, and don't accept it because a paper says so. Check it
— that's this notebook. As in Lesson 7, Week 5's models aren't refit here: Week 5
Lesson 8 already scored them on this exact test split, and its numbers are the bar.

## 2. Data

*Load, describe, and sanity-check the data before touching a model.*

Same dataset as every prior lesson that has touched churn — **Telco Customer
Churn**, one row per customer, account/contract/service features plus a binary
`Churn` label. `load_churn()` in `gbus564_coursework.utils.course_data` loads
and cleans it the same way every time.

In [ ]:
from gbus564_coursework.utils.course_data import load_churn

df = load_churn()
df.head()

In [ ]:
display(df.describe())

display(df.describe(exclude="number"))

### Data Prep

**Same test split as Week 5 Lesson 8** — same `test_size`, `random_state`, and
`stratify` — so every number below lines up against that scoreboard and Lesson 7's.
Unlike Lesson 7, there's no validation slice: TabPFN doesn't train on our data, so
there's nothing to early-stop. All 5,634 training rows go in as its context.

**Almost no preprocessing.** Lesson 7's network needed one-hot encoding and
scaling. TabPFN handles categorical and numeric columns itself — it was pretrained
on millions of synthetic tables with exactly that mix. The one step left is
stateless: mark the text columns as pandas `category` so TabPFN knows which columns
are categories rather than free text. Nothing here learns from the data, so there's
no `ColumnTransformer` to define.

In [ ]:
from sklearn.model_selection import train_test_split

X, y = df.drop(columns="Churn"), df["Churn"]
categorical_cols = X.select_dtypes(exclude="number").columns
X[categorical_cols] = X[categorical_cols].astype("category")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)
print(f"training rows (TabPFN's context): {len(X_train):,}   test rows: {len(X_test):,}")

## 3. Model

*Baseline first, then the challenger.*

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

**TabPFN.** Three settings, each with a reason:

- **Version 2 weights, explicitly.** The `tabpfn` package defaults to its newest
  model, but every release since 2.5 is non-commercial and won't download without a
  Prior Labs account and an accepted license (Lesson 8's "What it costs now").
  `create_default_for_version(ModelVersion.V2)` loads the open v2 weights instead —
  the peer-reviewed *Nature* model, and Telco's 7,043 rows sit well inside its
  10,000-row range.
- **`n_estimators=2`.** TabPFN averages several passes, each over a slightly different
  version of the table: columns in a new random order, numbers scaled a different way,
  and the two labels' internal numbering (churn = 0 or 1) swapped. Row order never
  matters (TabPFN reads rows as a set), but it's only *approximately* indifferent to
  these three, so averaging smooths them out — an ensemble, like Week 5's trees. Two
  passes is the minimum that covers both label orders and both scalings once; on this
  data the package's default of 8 scores within a thousandth of AUC and takes about 4×
  as long, so two keeps this notebook to a few minutes on a free Colab CPU.
- **The CPU guard.** On a CPU the package refuses more than 1,000 rows unless told
  otherwise, because prediction is slow there. The environment variable below lifts
  that guard. A Colab GPU (*Runtime → Change runtime type*) is faster but not needed.

Watch the two timings. `.fit()` is the step that took Lesson 7's network dozens of
epochs; here it only stores the training rows. The work happens in
`.predict_proba()`, where every test row attends to the training rows in one forward
pass — Lesson 8's *in-context learning*.

In [ ]:
%pip install -q tabpfn==9.2.0
import os
import time

os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"
from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion

tabpfn = TabPFNClassifier.create_default_for_version(
    ModelVersion.V2, n_estimators=2, device="auto", random_state=RANDOM_SEED,
)

start = time.perf_counter()
tabpfn.fit(X_train, y_train)
fit_seconds = time.perf_counter() - start

start = time.perf_counter()
tabpfn_proba = tabpfn.predict_proba(X_test)[:, 1]
predict_seconds = time.perf_counter() - start

print(f".fit():           {fit_seconds:6.1f} s  (stores the context rows)")
print(f".predict_proba(): {predict_seconds:6.1f} s  ({len(X_test):,} test rows, one forward pass)")

## 4. Evaluate

*Every model, scored against the same held-out test set — a metric only means
something next to what it is beating.*

**AUC** is the headline, as in Lesson 7: how well each model's scores *order*
customers by churn risk, which is what a call list needs. Accuracy at the 0.5 cutoff
is alongside it. The Week 5 and Lesson 7 rows are copied from those notebooks'
scoreboards, not recomputed: same test customers, so the comparison is like for like.

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score

scores = pd.DataFrame(
    {"test AUC": [roc_auc_score(y_test, tabpfn_proba), 0.5],
     "test accuracy": [accuracy_score(y_test, (tabpfn_proba > 0.5).astype(int)),
                       accuracy_score(y_test, baseline.predict(X_test))]},
    index=["TabPFN v2 (no training, no tuning)", "Baseline (majority class)"],
)
# Not refit — Week 5 Lesson 8's and Lesson 7's test scores on this identical split
earlier_scores = pd.DataFrame(
    {"test AUC": [0.834, 0.832, 0.827, 0.817], "test accuracy": [0.801, 0.791, 0.796, 0.787]},
    index=["XGBoost, tuned (Week 5 Lesson 8)", "Logistic regression (Week 5 Lesson 8)",
           "MLP — regularized (Lesson 7)", "MLP — plain (Lesson 7)"],
)
scores = pd.concat([scores, earlier_scores])
scores.sort_values("test AUC", ascending=False).round(3)

In [ ]:
auc = scores["test AUC"].drop("Baseline (majority class)").sort_values()
colors = ["#1A73E8" if name.startswith("TabPFN") else "#9AA0A6" for name in auc.index]
auc.plot.barh(color=colors, title="Test AUC by model (higher is better; gray = earlier lessons' scores)")
plt.xlim(0.80, 0.84)
plt.xlabel("AUC")
plt.tight_layout()

**Is the gap real?** The scoreboard ranks models to the third decimal, but 1,409
test customers is a modest sample. A **bootstrap** asks how much TabPFN's own AUC
would move if we'd happened to draw a different 1,409: resample the test rows with
replacement 1,000 times, recompute AUC each time, and look at the middle 95% of
those values. If that range is much wider than the gap to XGBoost, the gap is noise.

In [ ]:
n = len(y_test)
y_arr = y_test.to_numpy()
boot_auc = []
for _ in range(1000):
    idx = rng.integers(0, n, n)
    boot_auc.append(roc_auc_score(y_arr[idx], tabpfn_proba[idx]))
low, high = np.percentile(boot_auc, [2.5, 97.5])
print(f"TabPFN test AUC {scores.loc['TabPFN v2 (no training, no tuning)', 'test AUC']:.3f}, "
      f"95% bootstrap range {low:.3f} – {high:.3f}  (width {high - low:.3f})")

**Reading the scoreboard.** TabPFN tops the board at AUC 0.836 — with no training on
our data, no tuning, and almost no preprocessing. But the bootstrap range on its own
AUC runs from about 0.81 to 0.86, nearly 0.05 wide, and the 0.002 gap to Week 5's
tuned XGBoost is a small fraction of that. The honest verdict is a **tie** with the
best model this dataset has seen, not a win. On accuracy at the 0.5 cutoff, XGBoost
is slightly ahead (0.801 vs. 0.796) — also within noise, and a reminder that 0.5
isn't the business decision anyway (Week 2).

That's still a striking result, just not the one the headline promised. Week 5 spent
a whole lesson searching XGBoost's settings to reach 0.834; TabPFN got to the same
place in a single call with nothing tuned. And unlike Lesson 7's network, it didn't
need to give up a validation slice — all 5,634 training rows went in as context.

## 5. Interpret

*What is the model actually keying off of? Business-readable, not just
statistically significant.*

TabPFN has no coefficients (Week 3) or built-in importances (Week 5), and Lesson 7's
permutation importance would mean re-predicting the test set dozens of times — too
slow here. A cheaper, business-readable question: **what if?** Take 100 test
customers, set everyone's tenure to 1 month, then 6, 12, 24, 48, 72, and average the
predicted churn risk at each value. Do the same for contract type. All of those
what-if rows go through TabPFN in a single forward pass.

When tenure changes, `TotalCharges` is set to `MonthlyCharges × tenure` so the
what-if customer stays realistic — a 72-month customer with a one-month bill
total isn't someone the model has seen.

In [ ]:
sample = X_test.sample(100, random_state=RANDOM_SEED)
tenure_values = [1, 6, 12, 24, 48, 72]
contract_values = list(X_train["Contract"].cat.categories)

what_if = []
for t in tenure_values:
    what_if.append(sample.assign(tenure=t, TotalCharges=sample["MonthlyCharges"] * t,
                                 what="tenure", value=t))
for c in contract_values:
    rows = sample.copy()
    rows["Contract"] = pd.Categorical([c] * len(rows), categories=contract_values)
    what_if.append(rows.assign(what="contract", value=c))
what_if = pd.concat(what_if, ignore_index=True)

what_if["churn risk"] = tabpfn.predict_proba(what_if[X.columns])[:, 1]
by_tenure = what_if[what_if["what"] == "tenure"].groupby("value", sort=False)["churn risk"].mean()
by_contract = what_if[what_if["what"] == "contract"].groupby("value", sort=False)["churn risk"].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
by_tenure.plot(ax=axes[0], marker="o", color="#1A73E8",
               title="If these 100 customers had tenure of…")
axes[0].set_xlabel("Tenure (months)")
axes[0].set_ylabel("Average predicted churn risk")
by_contract.plot.bar(ax=axes[1], color="#1A73E8", rot=0,
                     title="If these 100 customers had contract…")
axes[1].set_xlabel("")
plt.tight_layout()
print(by_tenure.round(2).to_string(), "\n")
print(by_contract.round(2).to_string())

**Same story, told by a different model.** Moving the same 100 customers from 1 month
of tenure to 72 cuts their average predicted churn risk from about 47% to about 10%,
and moving them from month-to-month to a two-year contract cuts it from about 30% to
about 8%. That's what logistic regression's coefficients, XGBoost's SHAP values, and
Lesson 7's permutation importance all said: newer, month-to-month customers churn.
TabPFN found no new signal — which fits with it tying, not beating, the models that
found the same one.

This is also a first answer to Lesson 8's *"can we explain it?"* A what-if chart works
on any model, and a retention manager can read it. But it's an average over
customers, not an explanation of any one customer's score. For that you'd want SHAP
(Week 5 Lessons 10–11), which works on TabPFN too but is far slower than on trees:
every explanation means many more forward passes.

## 6. Recommend

**Recommendation:** keep Week 5's tuned XGBoost as the retention team's churn model.
TabPFN ties it (AUC 0.836 vs. 0.834, well inside the noise), so accuracy can't decide
— and on everything else XGBoost wins. It's already built, it scores a customer in a
fraction of a millisecond on any laptop, its SHAP explanations are exact and
near-instant (trees get their own fast SHAP algorithm), and it carries no license
questions. The v2 weights used here are open, but the
newer TabPFN releases are non-commercial: using them to inform this decision would
need a paid license (Lesson 8).

**Where TabPFN does earn a place: as a benchmark.** In a few minutes, with no tuning,
it showed roughly how good a churn ranking this data supports — the same level Week
5's tuning search reached. On the next tabular problem, run it early: if a quick
logistic regression already matches it, stop there; if there's a gap, that's roughly
how much a tuned model might buy. Lesson 8's rule held in both directions — the claim
wasn't dismissed, and it wasn't accepted on faith.

**Caveats / next steps:** one split, one seed, and two ensemble passes — differences
of a few thousandths of AUC are noise here. This is TabPFN v2; the newer releases
report stronger results but need a commercial license for business use. And the Week
5 and Lesson 7 rows are quoted, not refit, so there's no paired comparison on the same
predictions. Lesson 10 maps where networks do win outright: images, text, and
sequences.